# Datenbereinigung: Spend (Marketingkosten)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import help_130625_dam as h

DATA_PATH = os.path.join('..', 'Sources', 'Spend (Done).xlsx')
OUT_PATH  = os.path.join('..', 'data', 'cleaned', 'spend_clean.pkl')

pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True, precision=2)

## Laden und erste Inspektion

In [ ]:
df = pd.read_excel(DATA_PATH)

# Spaltennamen in snake_case umwandeln
df.columns = [h.to_snake(c) for c in df.columns]

print(f'Shape: {df.shape}')

n_before = df.shape[0]
h.descr_df(df, include='all', show_sample_rows=True)

In [ ]:
# Fehlende Werte
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
display(
    pd.DataFrame({'Missing': missing, '% Missing': missing_pct})
    .query('Missing > 0')
)
print('Zeilen ohne fehlende Werte:', df.dropna().shape[0])

In [ ]:
full_dupes = df.duplicated().sum()
print(f'Vollständige Duplikate: {full_dupes}')

# Spend hat keine ID — ein Duplikat ist eine exakte Übereinstimmung aller Felder
if full_dupes > 0:
    print('Beispiel für Duplikate:')
    dupes_df = df[df.duplicated(keep=False)].sort_values('date')
    display(dupes_df.head(10))
    
    print("\nZusammenfassung der numerischen Felder in Duplikaten (Summe):")
    # Berechne Summen nur für Zeilen, die entfernt werden (keep='first' behält eine, Rest wird berechnet)
    removed_dupes = df[df.duplicated(keep='first')]
    summary_dupes = removed_dupes[['impressions', 'spend', 'clicks']].sum()
    display(summary_dupes.to_frame('Summe der zu entfernenden Daten'))
    
    df = df.drop_duplicates().reset_index(drop=True)

print(f'Zeilen vorher: {n_before}  →  nachher: {len(df)}  (entfernt: {n_before - len(df)})')

## Datentypen: Datum

In [ ]:
DATE_COLS = ['date']

for col in DATE_COLS:
    # Explizite Angabe des Formats YYYY-MM-DD
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

# Ergebnis prüfen
print('Typen nach dem Parsen:')
print(df[DATE_COLS].dtypes)
print()

# NaT zählen = nicht erfolgreich geparst
for col in DATE_COLS:
    nat_count = df[col].isna().sum()
    print(f'{col}: NaT = {nat_count} ({nat_count/len(df)*100:.2f}%)')

## Numerische Felder

In [ ]:
# "Dreckige" Werte in numerischen Feldern prüfen
NUM_COLS = ['impressions', 'spend', 'clicks']

for col in NUM_COLS:
    print(f'--- {col} ---')
    print(f'  Typ: {df[col].dtype}')
    print(f'  Min: {df[col].min()}, Max: {df[col].max()}')
    neg = (df[col] < 0).sum()
    print(f'  Negative Werte: {neg}')

## Fehlende Werte

Dieser Datensatz ist die einzige Quelle für campaign, adgroup, ad, daher können sie nicht
nachgefüllt werden. Fehlende Werte werden durch "Unknown" ersetzt und der Typ in "category" geändert.

In [ ]:
# campaign, adgroup, ad — ~30% Lücken, mit 'Unknown' füllen
FILL_UNKNOWN = ['campaign', 'adgroup', 'ad']

for col in FILL_UNKNOWN:
    n_miss = df[col].isna().sum()
    df[col] = df[col].fillna('Unknown')
    print(f'{col}: {n_miss} Lücken gefüllt → "Unknown"')

# Prüfung
print('\nFehlende Werte nach dem Füllen:')
missing_after = df.isnull().sum()
display(
    pd.DataFrame({'Missing': missing_after, '% Missing': (missing_after / len(df) * 100).round(2)})
    .query('Missing > 0')
)

CAT_COLS = ['source', 'campaign', 'adgroup', 'ad']

for col in CAT_COLS:
    # Normalisierung: Leerzeichen an den Rändern entfernen
    df[col] = df[col].str.strip()
    df[col] = df[col].astype('category')
    print(f'{col}: {df[col].nunique()} eindeutige Werte')

## Zusammenfassung

In [ ]:
print(f'Finale Form des Datensatzes: {df.shape}')

h.descr_df(df, include='all', show_sample_rows=True)

## Speichern

In [ ]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_pickle(OUT_PATH)
df.to_excel(OUT_PATH.replace('.pkl', '.xlsx'))

# Zusammenfassung
summary_data = {
    'Metrik': [
        'Ursprüngliche Zeilen',
        'Zeilen nach Bereinigung',
        'Entfernte Duplikate',
        'Datumsbereich',
        'Kanäle (source)',
        'Gesamt Spend, €',
        'Lücken nach dem Füllen'
    ],
    'Wert': [
        n_before,
        len(df),
        n_before - len(df),
        f'{df["date"].min().date()} → {df["date"].max().date()}',
        df['source'].nunique(),
        f'{df["spend"].sum():,.2f}',
        df.isnull().sum().sum()
    ]
}

print(f'Gespeichert unter: {OUT_PATH}')
display(pd.DataFrame(summary_data))

## Statistik

In [ ]:
# Deskriptive Statistik für numerische Felder
# Berechnung von Mittelwert, Median, Modus und Bereich
num_stats = df[NUM_COLS].agg([
    'mean', 
    'median', 
    lambda x: x.mode()[0], 
    lambda x: x.max() - x.min()
]).T
num_stats.columns = ['Mean', 'Median', 'Mode', 'Range']

print("Deskriptive Statistik der numerischen Felder:")
display(num_stats)

# Analyse kategorialer Felder

print("\nVerteilung nach Quellen (Top 10):")
display(df['source'].value_counts().head(10).to_frame('Einträge'))

print("\nVerteilung nach Kampagnen (Top 10):")
display(df['campaign'].value_counts().head(10).to_frame('Einträge'))

## Datensatzbeschreibung

**Quelle:** `Spend.xlsx` — Daten aus Werbekonten  
**Zweck:** Erfassung der Marketingkosten zur Berechnung von ROI/ROAS und Analyse der Kanaleffizienz

| Spalte | Typ | Beschreibung |
|---|---|---|
| `date` | `datetime` | Datum der Ausgabe |
| `source` | `category` | Werbekanal/Quelle (Google, Facebook etc.) |
| `campaign` | `category` | Name der Werbekampagne |
| `impressions` | `int64` | Anzahl der Werbeeinblendungen (Impressions) |
| `spend` | `float64` | Tatsächliche Kosten in der Währung des Kontos |
| `clicks` | `int64` | Anzahl der Klicks |
| `adgroup` | `category` | Anzeigengruppe |
| `ad` | `category` | Konkrete Anzeige/Creative |

**Umfang:** 19.862 Einträge (nach Bereinigung), 8 Spalten  
**Wichtige Verknüpfungen:**
- `date` + `source` → Aggregation zum Abgleich mit Verkaufsergebnissen in `06_analytics`

917 Duplikate wurden entfernt, in denen fast keine Daten vorhanden waren. Datentypen wurden für die weitere Analyse konvertiert.